# OAE ensemble: pre-processing

This notebook turns the ROMS output of the six OAE ensemble members into the intermediate files that `oae_analysis.ipynb` plots. **Every file it writes goes to `out_dir`** (`/home/x-uheede/R/Intermediate_files/`), and `oae_analysis.ipynb` reads only from there. Nothing is written next to the notebook.

| Season | Members | Model output files |
|---|---|---|
| July | CDR1, CDR2, CDR3 | `Iceland3_MARBL_2024_{stream}.202407*.nc` |
| November | CDR4, CDR5, CDR6 | `Iceland3_MARBL_2024_{stream}.202411*.nc` |

Steps that read the full 3-D model output have one cell per season, so July and November can be run separately. With `overwrite = False`, a product whose file already exists is kept, so an interrupted run can be resumed.

**Run first:** `mixed_layer_depth.ipynb`, which writes the seasonal mixed-layer depth used in section 2.

| § | Writes (in `out_dir`) | Contents | Needs | Read by `oae_analysis.ipynb` |
|---|---|---|---|---|
| 1 | `Iceland3_cell_thickness_{July,Nov}2024_{ens}.nc` | layer thickness (time, s_rho, eta_rho, xi_rho) | his | — (used by §2) |
| 2 | `Iceland3_{ens}_mld_metrics.nc` | added-ALK inventory in the domain and above the MLD, and their ratio | bgc, §1, MLD file | §1 |
| 3 | `CDR{n}_cdr_metrics.nc` | roms-tools CDR efficiency and cumulative CO₂ uptake (flux- and DIC-based) | cstar | §1, §4, appendix |
| 4 | `ds_cstar{n}_regridded.nc` | air–sea CO₂ fluxes and column DIC inventories on a lat/lon grid | cstar | appendix (used by §5) |
| 5 | `Iceland3_CDR_cstar_cumsum_fields_{season}.nc` | cumulative CO₂ flux anomaly and DIC inventory anomaly vs. hours since release | §4 | §4 |
| 6 | `Iceland3_CDR_extrema_metrics_{season}.nc` | domain min pCO₂ anomaly, max pH, max ALK, surface ALK and DIC anomalies vs. hours since release | bgc, bgc_dia | §2, §3 |
| 6 | `Iceland3_1m_time0_96_mean_{season}.nc` | 0–96 h mean surface pCO₂ anomaly, and pH, ALK and DIC anomalies at 1 m | bgc, bgc_dia | §2 |
| 7 | `Iceland3_release_snapshots_{season}.nc` | ALK and pH anomalies at s_rho = 54 and surface pCO₂ anomaly, 6–48 h after release start | bgc, bgc_dia | §5 |

`{ens}` is CDR1…CDR6, `{n}` is 1…6 and `{season}` is July or November.

## Configuration

Paths, ensemble members and processing settings. The vertical grid (N, θs, θb, hc) is read from the grid file by roms-tools.

In [ ]:
import glob
import os

import numpy as np
import xarray as xr
from roms_tools import Grid, ROMSOutput
from roms_tools.vertical_coordinate import sigma_stretch

# =========================
# Paths (Anvil)
# =========================
model_grid_path = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments/Iceland3_MARBL_2024/P_INPUT/Iceland3_grid.nc"
experiment_dir = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments"
out_dir = "/home/x-uheede/R/Intermediate_files/"  # every file this notebook writes goes here

overwrite = False  # False: keep a product whose file already exists in out_dir

# =========================
# Ensemble members
# =========================
seasons = {
    "July": ["CDR1", "CDR2", "CDR3"],
    "November": ["CDR4", "CDR5", "CDR6"],
}
run_month = {"July": "202407", "November": "202411"}  # month in the model output file names

# =========================
# Processing settings
# =========================
target_depth_levels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]  # m, depth levels for roms-tools regridding (§4, §6)
time_rel_common = np.arange(-48, 240, 1)  # h since release start, common time axis of the extrema (§6)
mean_window_hours = (0, 96)  # h since release start, averaging window of the 1 m mean fields (§6)
cumsum_hours = np.arange(0, 241, 1)  # h since release start, time axis of the cumulative fields (§5)

# Seasonal mixed-layer depth (Stevens 2010), written by mixed_layer_depth.ipynb (§2)
mld_file = os.path.join(out_dir, "Iceland3_mld_stevens2010.nc")

# Release snapshots (§7): release start in hours after each member's first model record
snapshot_release_start = {"CDR1": 36.0, "CDR2": 70.0, "CDR3": 70.0, "CDR4": 48.0, "CDR5": 55.0, "CDR6": 48.0}
snapshot_hours_after_release = [6, 12, 24, 48]
snapshot_s_rho = 54  # model level of the ALK and pH snapshots (s_rho runs from 0 at the bottom up)

### Shared helpers

File locations, the roms-tools grid, and the time handling shared by all sections. Consecutive model output files share one record, so `remove_duplicate_time` keeps each time stamp once.

In [ ]:
os.makedirs(out_dir, exist_ok=True)
season_of = {ens: season for season, members in seasons.items() for ens in members}

roms_grid = Grid._from_file(model_grid_path)  # roms-tools grid; vertical coordinate read from the file
grid_ds = xr.open_dataset(model_grid_path)
cell_area = 1 / (grid_ds["pm"] * grid_ds["pn"])  # m2


def model_files(ens, stream):
    """Sorted output files of one member; stream is 'his', 'bgc', 'bgc_dia' or 'cstar'."""
    month = run_month[season_of[ens]]
    pattern = f"{experiment_dir}/Iceland3_MARBL_2024_{ens}/Iceland3_MARBL_2024_{stream}.{month}*.nc"
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files match {pattern}")
    return files


def open_model(ens, stream, **kwargs):
    """Lazily open all output files of one member and stream along time."""
    return xr.open_mfdataset(
        model_files(ens, stream),
        combine="nested",
        concat_dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        **kwargs,
    )


def out_path(name):
    return os.path.join(out_dir, name)


def thickness_file(ens):
    month = {"July": "July2024", "November": "Nov2024"}[season_of[ens]]
    return out_path(f"Iceland3_cell_thickness_{month}_{ens}.nc")


def cstar_regridded_file(ens):
    return out_path(f"ds_cstar{ens.removeprefix('CDR')}_regridded.nc")


def should_write(path):
    """False, with a note, if `path` already exists and overwrite is off."""
    if os.path.exists(path) and not overwrite:
        print(f"  keeping existing {os.path.basename(path)} (set overwrite = True to recompute)")
        return False
    return True


def remove_duplicate_time(ds):
    """Keep one record per time stamp (consecutive output files share a record)."""
    _, index = np.unique(ds["time"], return_index=True)
    return ds.isel(time=index)


def add_relative_time(ds, t0):
    """Make hours since t0 ('time_rel') the time dimension and drop the absolute time."""
    time_rel = (ds["time"] - t0) / np.timedelta64(1, "h")
    ds = ds.assign_coords(time_rel=("time", time_rel.data)).swap_dims({"time": "time_rel"})
    return ds.drop_vars("time")


print(f"Writing to {out_dir}")
print(f"Grid: N = {roms_grid.N}, theta_s = {roms_grid.theta_s}, theta_b = {roms_grid.theta_b}, hc = {roms_grid.hc}")

## 1. Cell thickness

Vertical thickness of every rho cell (m) at every history record, from the free surface `zeta` and the grid's s-coordinate (Vtransform = 2). Land cells are NaN. Section 2 uses it for cell volumes and for the depth of each cell below the surface.

**Writes** `Iceland3_cell_thickness_July2024_{CDR1-3}.nc` and `Iceland3_cell_thickness_Nov2024_{CDR4-6}.nc` (float32, one file per member). These are large, roughly one 3-D field per history record.

In [ ]:
def write_cell_thickness(ens):
    """Thickness (m) of every rho cell, from the free surface in the history files."""
    path = thickness_file(ens)
    if not should_write(path):
        return

    # s-coordinate at the cell faces (dimension s_w, bottom -> top)
    cs_w, sigma_w = sigma_stretch(roms_grid.theta_s, roms_grid.theta_b, roms_grid.N, "w")
    hc = roms_grid.hc
    h = grid_ds["h"]

    zeta = open_model(ens, "his", chunks={"time": 1})["zeta"]
    z_w = zeta + (zeta + h) * (hc * sigma_w + h * cs_w) / (hc + h)  # height of the cell faces (Vtransform = 2)

    thickness = z_w.diff("s_w").rename({"s_w": "s_rho"}).where(grid_ds["mask_rho"] == 1)
    thickness = thickness.transpose("time", "s_rho", "eta_rho", "xi_rho")
    thickness.attrs = {"units": "m", "long_name": "vertical cell thickness (Hz)"}

    thickness.to_dataset(name="cell_thickness").to_netcdf(
        path, encoding={"cell_thickness": {"dtype": "float32"}}
    )
    print(f"  saved {os.path.basename(path)}")

In [ ]:
# July: CDR1-3
for ens in seasons["July"]:
    print(ens)
    write_cell_thickness(ens)

In [ ]:
# November: CDR4-6
for ens in seasons["November"]:
    print(ens)
    write_cell_thickness(ens)

## 2. Added-ALK inventory and fraction in the mixed layer

For every history record:
- `total_alk`: the domain inventory of added alkalinity, Σ (ALK − ALK_ALT_CO2) · cell volume (mol);
- `upper_alk_inventory`: the same sum over cells whose lower face lies above the mixed-layer depth;
- `mld_alk_fraction`: their ratio.

The mixed-layer depth is the **seasonal Stevens (2010) MLD** from `mixed_layer_depth.ipynb` (`Iceland3_mld_stevens2010.nc`, case `July` or `November`). The values used for the initial submission were 2.5 m (July) and 5.8 m (November). The cell prints the values it uses.

The depth of each cell's lower face is the cumulative thickness from the surface. s_rho runs from 0 (bottom) to N−1 (top), so the cumulative sum is taken over the reversed level order and then reversed back. This is done by position, so it does not depend on whether the files carry an `s_rho` coordinate.

Thickness (§1) and bgc records are matched by position, so the cell checks that their record counts agree.

An existing file is recomputed whenever its `target_depth_m` differs from the seasonal MLD, even with `overwrite = False`. This covers files made with the hard-coded 2.5 / 5.8 m.

**Writes** `Iceland3_{ens}_mld_metrics.nc` for CDR1–6. **Needs** §1 and `mixed_layer_depth.ipynb`.

In [ ]:
def seasonal_mld():
    """Upper-layer depth (m) for every member: the seasonal Stevens (2010) MLD."""
    if not os.path.exists(mld_file):
        raise FileNotFoundError(f"{mld_file} not found: run mixed_layer_depth.ipynb first")
    with xr.open_dataset(mld_file) as ds_mld:
        return {ens: float(ds_mld["mld"].sel(case=season)) for ens, season in season_of.items()}


def stored_depth(path):
    """Upper-layer depth (m) an existing mld_metrics file was computed for."""
    with xr.open_dataset(path) as ds:
        return float(ds["target_depth_m"])


def write_mld_metrics(ens, upper_depth):
    """Added-ALK inventory in the domain and above `upper_depth` (m below the surface)."""
    path = out_path(f"Iceland3_{ens}_mld_metrics.nc")
    if os.path.exists(path) and not np.isclose(stored_depth(path), upper_depth):
        print(f"  {os.path.basename(path)} was computed for {stored_depth(path):.2f} m: recomputing")
    elif not should_write(path):
        return

    ds_bgc = open_model(ens, "bgc", chunks={"time": 1})
    alk_anomaly = ds_bgc["ALK"] - ds_bgc["ALK_ALT_CO2"]  # mmol m-3

    thickness = xr.open_dataset(thickness_file(ens), chunks={"time": 1})["cell_thickness"]
    if thickness.sizes["time"] != alk_anomaly.sizes["time"]:
        raise ValueError(
            f"{ens}: {thickness.sizes['time']} thickness records but {alk_anomaly.sizes['time']} bgc records"
        )
    thickness = thickness.drop_vars("time", errors="ignore")  # matched to the bgc records by position

    volume = thickness * cell_area  # m3

    # Depth of each cell's lower face below the surface: cumulative thickness from the top
    # level (s_rho = N-1) down, computed on the reversed level order and reversed back.
    top_down = slice(None, None, -1)
    depth_lower_face = thickness.isel(s_rho=top_down).cumsum("s_rho").isel(s_rho=top_down)
    volume_upper = volume.where(depth_lower_face <= upper_depth, 0)

    sum_dims = ["s_rho", "eta_rho", "xi_rho"]
    total_alk = (alk_anomaly * volume).sum(sum_dims) * 1e-3  # mmol -> mol
    upper_alk = (alk_anomaly * volume_upper).sum(sum_dims) * 1e-3

    xr.Dataset(
        data_vars={
            "total_alk": total_alk,
            "upper_alk_inventory": upper_alk,
            "mld_alk_fraction": upper_alk / total_alk,
            "target_depth_m": xr.DataArray(upper_depth),
        },
        attrs={
            "ensemble": ens,
            "target_depth_meters": upper_depth,
            "units_total_inventory": "moles",
            "description": f"Added-ALK inventory and its fraction above the seasonal MLD ({upper_depth:.2f} m)",
        },
    ).to_netcdf(path)
    print(f"  saved {os.path.basename(path)}")


upper_depths = seasonal_mld()

In [ ]:
# July: CDR1-3
for ens in seasons["July"]:
    print(f"{ens}: upper-layer depth {upper_depths[ens]:.2f} m")
    write_mld_metrics(ens, upper_depths[ens])

In [ ]:
# November: CDR4-6
for ens in seasons["November"]:
    print(f"{ens}: upper-layer depth {upper_depths[ens]:.2f} m")
    write_mld_metrics(ens, upper_depths[ens])

## 3. CDR metrics (roms-tools)

`ROMSOutput.cdr_metrics()` on the cstar output, with duplicate records removed. It writes the CDR efficiency and cumulative carbon uptake computed both from the air–sea flux difference and from the DIC difference. roms-tools also shows a check plot per member. This takes roughly 15 minutes per member.

**Writes** `CDR{n}_cdr_metrics.nc` (`cdr_efficiency`, `co2_uptake`, `cdr_efficiency_from_dic`, `co2_uptake_from_dic`) for CDR1–6.

In [ ]:
def write_cdr_metrics(ens):
    """roms-tools CDR efficiency and cumulative CO2 uptake (flux- and DIC-based)."""
    path = out_path(f"{ens}_cdr_metrics.nc")
    if not should_write(path):
        return

    roms_output = ROMSOutput(grid=roms_grid, path=model_files(ens, "cstar"), use_dask=True)
    roms_output.ds = remove_duplicate_time(roms_output.ds)
    roms_output.cdr_metrics()
    ds_cdr = roms_output.ds_cdr

    xr.Dataset(
        data_vars={
            "cdr_efficiency": ds_cdr["cdr_efficiency_from_flux"],
            "co2_uptake": ds_cdr["cdr_carbon_uptake_from_flux"],
            "cdr_efficiency_from_dic": ds_cdr["cdr_efficiency_from_DIC_difference"],
            "co2_uptake_from_dic": ds_cdr["cdr_carbon_uptake_from_DIC_difference"],
        },
        attrs={
            "ensemble": ens,
            "description": f"roms-tools CDR efficiency and cumulative carbon uptake for {ens}",
        },
    ).to_netcdf(path)
    print(f"  saved {os.path.basename(path)}")

In [ ]:
# July: CDR1-3
for ens in seasons["July"]:
    print(ens)
    write_cdr_metrics(ens)

In [ ]:
# November: CDR4-6
for ens in seasons["November"]:
    print(ens)
    write_cdr_metrics(ens)

## 4. Regridded cstar fields

Column DIC inventories `hDIC_sum` and `hDIC_ALT_CO2_sum` (`hDIC` summed over s_rho, mmol m⁻²) and the air–sea CO₂ fluxes `FG_CO2` and `FG_ALT_CO2`. Duplicate records are removed and the fields are regridded with roms-tools to a regular lat/lon grid.

**Writes** `ds_cstar{n}_regridded.nc` for CDR1–6.

In [ ]:
def write_cstar_regridded(ens):
    """Air-sea CO2 fluxes and column DIC inventories, regridded to lat/lon."""
    path = cstar_regridded_file(ens)
    if not should_write(path):
        return

    roms_cstar = ROMSOutput(grid=roms_grid, path=model_files(ens, "cstar"), use_dask=True)
    ds = remove_duplicate_time(roms_cstar.ds)
    ds["hDIC_sum"] = ds["hDIC"].sum("s_rho", skipna=True)
    ds["hDIC_ALT_CO2_sum"] = ds["hDIC_ALT_CO2"].sum("s_rho", skipna=True)
    roms_cstar.ds = ds

    ds_regridded = roms_cstar.regrid(
        depth_levels=target_depth_levels,
        var_names=["hDIC_sum", "hDIC_ALT_CO2_sum", "FG_CO2", "FG_ALT_CO2"],
    )
    ds_regridded.to_netcdf(path)
    print(f"  saved {os.path.basename(path)}")

In [ ]:
# July: CDR1-3
for ens in seasons["July"]:
    print(ens)
    write_cstar_regridded(ens)

In [ ]:
# November: CDR4-6
for ens in seasons["November"]:
    print(ens)
    write_cstar_regridded(ens)

## 5. Cumulative air–sea CO₂ flux and DIC inventory anomalies

From the regridded cstar fields (§4), for each member:
- t = 0 is the first time an air–sea CO₂ flux anomaly |FG_CO2 − FG_ALT_CO2| > 1e-9 appears anywhere in the domain;
- fields are interpolated to hourly steps 0–240 h after release;
- `FG_CO2_cumsum_anom` is the time-integrated flux anomaly (mmol m⁻²);
- `hDIC_anom` is the column DIC inventory anomaly (mmol m⁻²).

Members are stacked along `ens`, one file per season. This step is light: it only reads §4's 2-D fields.

**Writes** `Iceland3_CDR_cstar_cumsum_fields_July.nc` and `…_November.nc`. **Needs** §4.

In [ ]:
def flux_start_time(ds, threshold=1e-9):
    """First time an air-sea CO2 flux anomaly appears anywhere in the domain."""
    signal = np.abs(ds["FG_CO2"] - ds["FG_ALT_CO2"]).max(dim=("lat", "lon")).load()
    return signal.where(signal > threshold, drop=True).time.min()


def write_cstar_cumsum(season):
    """Cumulative FG_CO2 anomaly and DIC inventory anomaly vs. hours since release, per season."""
    path = out_path(f"Iceland3_CDR_cstar_cumsum_fields_{season}.nc")
    if not should_write(path):
        return

    members, t0s = [], []
    for ens in seasons[season]:
        ds = remove_duplicate_time(xr.open_dataset(cstar_regridded_file(ens)))
        t0 = flux_start_time(ds)
        ds = add_relative_time(ds, t0).sel(time_rel=slice(0, None)).interp(time_rel=cumsum_hours)

        fg_anom = ds["FG_CO2"] - ds["FG_ALT_CO2"]  # mmol m-2 s-1
        members.append(
            xr.Dataset(
                {
                    "FG_CO2_cumsum_anom": fg_anom.cumsum("time_rel") * 3600.0,  # hourly steps -> mmol m-2
                    "hDIC_anom": ds["hDIC_sum"] - ds["hDIC_ALT_CO2_sum"],  # mmol m-2
                }
            ).load()
        )
        t0s.append(t0.values)
        print(f"  {ens}: t = 0 at {t0.values}")

    ds_ens = xr.concat(members, dim="ens").assign_coords(ens=seasons[season])
    ds_ens["release_start_time"] = ("ens", np.array(t0s))
    ds_ens["FG_CO2_cumsum_anom"].attrs = {
        "units": "mmol/m^2",
        "long_name": "Cumulative air-sea CO2 flux anomaly (time integrated)",
    }
    ds_ens["hDIC_anom"].attrs = {"units": "mmol/m^2", "long_name": "hDIC inventory anomaly (vertically integrated)"}
    ds_ens["release_start_time"].attrs = {"long_name": "first time an air-sea CO2 flux anomaly appears (t = 0)"}
    ds_ens.attrs = {
        "description": f"Cumulative FG_CO2 and hDIC anomalies per grid cell, {season} ensemble, aligned at t = 0",
        "model": "Iceland3 MARBL 2024",
    }
    ds_ens.to_netcdf(path)
    print(f"  saved {os.path.basename(path)}")


for season in seasons:
    print(season)
    write_cstar_cumsum(season)

## 6. Ensemble extrema and 0–96 h mean fields

The bgc (ALK, DIC and their `_ALT_CO2` twins) and bgc_dia (pH, surface pCO₂) output of each member is regridded with roms-tools to lat/lon and `target_depth_levels`. Steps:
- t = 0 is the first time the 1 m ALK anomaly exceeds 1e-6 anywhere;
- all members are interpolated to hourly steps −48…239 h (`time_rel_common`) and stacked along `ens`.

Two files per season:
- **Extrema** (`Iceland3_CDR_extrema_metrics_{season}.nc`), time series of:
  - `min_pCO2SURF_anom`: domain minimum of the surface pCO₂ anomaly;
  - `max_pH`: domain maximum of pH (absolute pH, not an anomaly);
  - `max_ALK_anom`, `max_ALK_surface_anom` (1 m) and `max_DIC_anom`: domain maxima of the anomalies.
- **1 m mean fields** (`Iceland3_1m_time0_96_mean_{season}.nc`): the 0–96 h mean of the surface pCO₂ anomaly (`pCO2SURF_anom`), pH at 1 m (`pH`), and the ALK and DIC anomalies at 1 m (`ALK_anom`, `DIC_anom`).

This is the heaviest step: it reads the full 3-D bgc and bgc_dia output.

In [ ]:
def regridded_member(ens, stream, var_names):
    """One output stream of a member, duplicate records removed, regridded to lat/lon/depth."""
    roms_output = ROMSOutput(grid=roms_grid, path=model_files(ens, stream), use_dask=True)
    roms_output.ds = remove_duplicate_time(roms_output.ds)
    return roms_output.regrid(depth_levels=target_depth_levels, var_names=var_names)


def alk_start_time(ds_bgc, threshold=1e-6):
    """First time the ALK anomaly at the shallowest depth level appears anywhere in the domain."""
    alk_1m = ds_bgc["ALK"].isel(depth=0).load() - ds_bgc["ALK_ALT_CO2"].isel(depth=0).load()
    signal = alk_1m.max(dim=("lat", "lon"))
    return signal.where(signal > threshold, drop=True).time.min()


def write_season_fields(season):
    """Ensemble extrema time series and 0-96 h mean 1 m fields for one season."""
    extrema_path = out_path(f"Iceland3_CDR_extrema_metrics_{season}.nc")
    mean_path = out_path(f"Iceland3_1m_time0_96_mean_{season}.nc")
    todo = [path for path in (extrema_path, mean_path) if should_write(path)]
    if not todo:
        return

    bgc, dia, t0s = [], [], []
    for ens in seasons[season]:
        ds_bgc = regridded_member(ens, "bgc", ["ALK", "ALK_ALT_CO2", "DIC", "DIC_ALT_CO2"])
        ds_dia = regridded_member(ens, "bgc_dia", ["pH_3D", "pCO2SURF", "pCO2SURF_ALT_CO2"])
        t0 = alk_start_time(ds_bgc)
        print(f"  {ens}: t = 0 at {t0.values}")
        t0s.append(t0.values)
        bgc.append(add_relative_time(ds_bgc, t0).interp(time_rel=time_rel_common))
        dia.append(add_relative_time(ds_dia, t0).interp(time_rel=time_rel_common))

    ds_bgc = xr.concat(bgc, dim="ens").assign_coords(ens=seasons[season])
    ds_dia = xr.concat(dia, dim="ens").assign_coords(ens=seasons[season])
    release_start = xr.DataArray(
        np.array(t0s),
        dims="ens",
        coords={"ens": seasons[season]},
        attrs={"long_name": "first time the 1 m ALK anomaly exceeds 1e-6 (t = 0)"},
    )

    pco2_anom = ds_dia["pCO2SURF"] - ds_dia["pCO2SURF_ALT_CO2"]
    ph = ds_dia["pH_3D"]
    alk_anom = ds_bgc["ALK"] - ds_bgc["ALK_ALT_CO2"]  # mmol m-3
    dic_anom = ds_bgc["DIC"] - ds_bgc["DIC_ALT_CO2"]  # mmol m-3

    if extrema_path in todo:
        ds_extrema = xr.Dataset(
            {
                "min_pCO2SURF_anom": pco2_anom.min(["lat", "lon"]),
                "max_pH": ph.max(["depth", "lat", "lon"]),
                "max_ALK_anom": alk_anom.max(["depth", "lat", "lon"]),
                "max_ALK_surface_anom": alk_anom.isel(depth=0).max(["lat", "lon"]),
                "max_DIC_anom": dic_anom.max(["depth", "lat", "lon"]),
                "release_start_time": release_start,
            },
            attrs={
                "description": f"Domain extrema vs. hours since release start, {season} ensemble",
                "model": "Iceland3 MARBL 2024",
            },
        )
        ds_extrema.load().to_netcdf(extrema_path)
        print(f"  saved {os.path.basename(extrema_path)}")

    if mean_path in todo:
        window = slice(*mean_window_hours)

        def mean_at_1m(da):
            return da.sel(depth=1, method="nearest").sel(time_rel=window).mean("time_rel")

        ds_mean = xr.Dataset(
            {
                "pCO2SURF_anom": pco2_anom.sel(time_rel=window).mean("time_rel"),
                "pH": mean_at_1m(ph),
                "ALK_anom": mean_at_1m(alk_anom),
                "DIC_anom": mean_at_1m(dic_anom),
                "release_start_time": release_start,
            },
            attrs={
                "description": (
                    f"Mean over {mean_window_hours[0]}-{mean_window_hours[1]} h after release start: "
                    f"surface pCO2 anomaly, and pH, ALK and DIC anomalies at 1 m, {season} ensemble"
                ),
                "model": "Iceland3 MARBL 2024",
            },
        )
        ds_mean.load().to_netcdf(mean_path)
        print(f"  saved {os.path.basename(mean_path)}")

In [ ]:
# July: CDR1-3
write_season_fields("July")

In [ ]:
# November: CDR4-6
write_season_fields("November")

## 7. Release snapshots

For each member, the model record nearest to 6, 12, 24 and 48 h after the release start (`snapshot_release_start`, hours after the member's first record) provides:
- `ALK_anom`: ALK − ALK_ALT_CO2 at s_rho = 54;
- `pH_anom`: pH_3D − pH_3D_ALT_CO2 at s_rho = 54;
- `pCO2SURF_anom`: the surface pCO₂ anomaly.

Each file also stores the model hour of every snapshot and the grid's lon/lat/mask, so `oae_analysis.ipynb` can map the snapshots without opening model output. Only four records per member are read, so this step is light.

**Writes** `Iceland3_release_snapshots_July.nc` and `…_November.nc`.

In [ ]:
def nearest_records(ds, target_hours):
    """Indices and model hours of the records nearest to `target_hours` after the first record."""
    seconds = ds["ocean_time"].values
    hours = (seconds - seconds[0]) / 3600.0
    index = [int(np.argmin(np.abs(hours - t))) for t in target_hours]
    return index, hours[index]


def at_records(da, index, **level):
    """Select records (and optionally a level), dropping coordinates so members line up."""
    da = da.isel(time=index, **level).reset_coords(drop=True)
    return da.drop_vars("time", errors="ignore")


def write_snapshots(season):
    """ALK and pH anomalies at s_rho = snapshot_s_rho and surface pCO2 anomaly after release start."""
    path = out_path(f"Iceland3_release_snapshots_{season}.nc")
    if not should_write(path):
        return

    members = []
    for ens in seasons[season]:
        ds_bgc = open_model(ens, "bgc", decode_times=False)
        ds_dia = open_model(ens, "bgc_dia", decode_times=False)
        target_hours = snapshot_release_start[ens] + np.asarray(snapshot_hours_after_release, dtype=float)
        i_bgc, hours_bgc = nearest_records(ds_bgc, target_hours)
        i_dia, hours_dia = nearest_records(ds_dia, target_hours)

        snap = xr.Dataset(
            {
                "ALK_anom": at_records(ds_bgc["ALK"] - ds_bgc["ALK_ALT_CO2"], i_bgc, s_rho=snapshot_s_rho),
                "pH_anom": at_records(ds_dia["pH_3D"] - ds_dia["pH_3D_ALT_CO2"], i_dia, s_rho=snapshot_s_rho),
                "pCO2SURF_anom": at_records(ds_dia["pCO2SURF"] - ds_dia["pCO2SURF_ALT_CO2"], i_dia),
                "model_hour_bgc": ("time", hours_bgc),
                "model_hour_dia": ("time", hours_dia),
            }
        )
        snap = snap.rename_dims(time="hours_after_release")
        members.append(snap.load())
        print(f"  {ens}: records at model hours {np.round(hours_dia, 1)}")

    horizontal = ("eta_rho", "xi_rho")
    ds_snap = xr.concat(members, dim="ens").assign_coords(
        ens=seasons[season],
        hours_after_release=snapshot_hours_after_release,
        lon_rho=(horizontal, grid_ds["lon_rho"].values),
        lat_rho=(horizontal, grid_ds["lat_rho"].values),
    )
    ds_snap["mask_rho"] = (horizontal, grid_ds["mask_rho"].values)
    ds_snap["release_start_hour"] = ("ens", [snapshot_release_start[ens] for ens in seasons[season]])
    ds_snap["ALK_anom"].attrs = {"units": "meq/m^3", "long_name": f"ALK anomaly at s_rho = {snapshot_s_rho}"}
    ds_snap["pH_anom"].attrs = {"long_name": f"pH anomaly at s_rho = {snapshot_s_rho}"}
    ds_snap["pCO2SURF_anom"].attrs = {"units": "ppmv", "long_name": "surface pCO2 anomaly"}
    ds_snap.attrs = {
        "description": f"Snapshots after release start, {season} ensemble",
        "s_rho": snapshot_s_rho,
    }
    ds_snap.to_netcdf(path)
    print(f"  saved {os.path.basename(path)}")


for season in seasons:
    print(season)
    write_snapshots(season)

## Changes from the original notebook

The original version of this notebook is kept outside the repository, in `archive/notebooks/OAE_ensemble.ipynb`.

**Files and coverage**
- Every product is written to `out_dir` (previously several went to the notebook folder), for all six members.
- Seasonal files replace the old suffixes: `…_July.nc` / `…_November.nc` instead of no suffix / `_pt2`.
- The 1 m mean fields are one file per season with named variables, instead of four files with `__xarray_dataarray_variable__`.
- Thickness files for CDR4–6 are named `Nov2024`; they were mislabelled `July2024`.

**Methods**
- The ALK-fraction depth is read from `mixed_layer_depth.ipynb`'s output instead of being hard-coded (2.5 m / 5.8 m).
- The ALK-fraction depth mask is computed by position, independent of an `s_rho` coordinate.
- The release snapshots are written as data (§7); their maps are drawn in `oae_analysis.ipynb` §5.
- The roms-tools grid is read with `Grid._from_file`, with no separate vertical-coordinate update.

**Not carried over**
- The interrupted single-member `cdr_metrics` cell, which section 3 supersedes.
- The two cells plotting an undefined `mld_timeseries`.
- The T/S profile cell, whose code had been deleted.
- Duplicate helper definitions and empty cells.